## Notebook Description: Data Curation and Averaging of Gas and Nutrition Data

This notebook outlines a comprehensive data curation and preprocessing workflow for two datasets: 'gas' and 'nutrition'. The primary objectives include standardizing categorical variables, filtering out erroneous or incomplete records, and aggregating replicate measurements. Key methodologies involve leveraging the pandas library for data manipulation, including the use of `df.replace()` for consistent categorization (e.g., 'Breding ' to 'Breeding'), `df.str.contains()` and `df.isna()` for identifying and marking rows for deletion based on specific keywords in remark columns or missing 'methane_intensity' values, and `df.groupby().mean()` for calculating average values and replicate counts (e.g., $\text{n_replicates_gas}$, $\text{n_replicates_nutrition}$) for each unique sample identifier ($\text{id_lab}$). The outcome is a set of cleaned and averaged datasets ready for further analysis, specifically $\text{gas_clean_av2}$ and $\text{nutrition_av2}$, which condense experimental replicates into single, representative entries.

# 1.0 Libraries

In [66]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [67]:
import numpy as np
import pandas as pd

# 2.0 Import data

In [68]:
gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/subsets_1_2_3_4_gas_sorted_by_sql.csv')
nutrition = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/subsets_1_2_3_4_nutrition_sorted_by_sql.csv')

In [69]:
gas.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2
0,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,33.240646,0,95.43754089,34.829738,Standar,NaN,NaN,NaN,NaN,NaN
1,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,32.485469,0,89.75200777,36.194699,Standar,NaN,NaN,NaN,NaN,NaN


In [70]:
gas.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2'],
      dtype='object')

In [71]:
nutrition.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,1,1.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,91.930807,12.725691,87.274309,33.010522,28.489614,45.210329
1,1,2.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,92.010000,12.629062,87.370938,33.010522,27.612657,45.235119


# 3.0 Formatting

## 3.1 Drop duplicates

In [72]:
gas.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2'],
      dtype='object')

In [73]:
key = ['id_lab', 'batch', 'run', 'replication', 'syrange']

In [74]:
duplicates = gas[
    gas.duplicated(subset=key, keep=False)
].sort_values(key)

In [75]:
print(len(duplicates))
duplicates.iloc[:-10, :20]

1967


,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,batch,run,replication,syrange,sample_weight_g,undigested_dm_g,dm_incubated,digested_feed_mg
549,1,406,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,1,1.0,0.5002,0.2082,462.036651,253.836651
550,1,406,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,1,1.0,0.5002,0.2082,462.036651,253.836651
551,1,407,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,2,2.0,0.5001,0.2150,461.944280,246.944280
552,1,407,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,2,2.0,0.5001,0.2150,461.944280,246.944280
553,1,408,Genetic bank,F24-3426,CIAT-989,Fabales,Fabaceae,Indigofera,hirsuta,Indigofera hirsuta,Herbaceous_legumes,5,13,1,3,3.0,0.5,0.2217,461.851910,240.151910
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5957,3,426,Genetic bank,F25-1667,CIAT-9578,Fabales,Fabaceae,Dioclea,sericea,Dioclea sericea,Herbaceous_legumes,24,50,2,1,111.0,NaN,NaN,NaN,NaN
5958,3,427,Genetic bank,F25-1667,CIAT-9578,Fabales,Fabaceae,Dioclea,sericea,Dioclea sericea,Herbaceous_legumes,24,50,2,2,112.0,NaN,0.3240,NaN,NaN
5959,3,427,Genetic bank,F25-1667,CIAT-9578,Fabales,Fabaceae,Dioclea,sericea,Dioclea sericea,Herbaceous_legumes,24,50,2,2,112.0,NaN,0.3240,NaN,NaN
5960,3,428,Genetic bank,F25-1667,CIAT-9578,Fabales,Fabaceae,Dioclea,sericea,Dioclea sericea,Herbaceous_legumes,24,50,2,3,113.0,NaN,0.3290,NaN,NaN


## 3.2 Requisitioner update

In [76]:
gas.requisitioner.unique()

array(['LMF', 'Genetic bank', 'Breeding', nan, 'Set 2', 'Breding ',
       'LMF-invivo', 'Benchmark', 'Genbank', 'Isabel Molina',
       'Mauricio Sotelo', 'Jacobo Arango/Alejandro Montoya',
       'Jacobo Arango/ Alejandro Montoya'], dtype=object)

In [77]:
# Define the mapping for requisitioner
requisitioner_mapping = {
    'Breding ': 'Breeding',
    'Genbank': 'Genetic_bank',
    'Genetic bank': 'Genetic_bank'
}

# Apply the mapping to centers datt
gas['requisitioner'] = gas['requisitioner'].replace(requisitioner_mapping)
nutrition['requisitioner'] = nutrition['requisitioner'].replace(requisitioner_mapping)

## 3.3 Urocloa Taxonomy update


In [78]:
# Define the mapping for tax name
tax_name_mapping = {
    'Brachiaria humidicola': 'Urochloa humidicola',
    'Brachiaria interespecifico': 'Urochloa interespecific',
    'Brachiaria interespecifico ': 'Urochloa interespecific',
    'Urochloa interespecifico': 'Urochloa interespecific'

}

# Apply the mapping to centers datt
gas['tax_name'] = gas['tax_name'].replace(tax_name_mapping)
nutrition['tax_name'] = nutrition['tax_name'].replace(tax_name_mapping)

In [79]:
# Define the mapping for genus
genus_mapping = {
    'Brachiaria': 'Urochloa'
}

# Apply the mapping to centers datt
gas['genus'] = gas['genus'].replace(genus_mapping)
nutrition['genus'] = nutrition['genus'].replace(genus_mapping)

In [80]:
# Define the mapping for species
species_mapping = {
    'interespecifico': 'interespecific',
    ' interespecifico': 'interespecific',
}

# Apply the mapping to centers datt
gas['species'] = gas['species'].replace(species_mapping)
nutrition['species'] = nutrition['species'].replace(species_mapping)

## 3.4 Repeated samples

In [81]:
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1651"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1652"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1653"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1654"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1655"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1656"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1658"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1664"))]
gas = gas[~((gas["subset"] == 2) & (gas["id_lab"] == "F25-1665"))]

# 4.0 Curation

In [82]:
remarks_columns = ['information_remarks_1','information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
                   'digest_remarks_1', 'digest_remarks_2']

# Convert columns to string type and check for 'elim' or 'descar' (case-insensitive)
conditions = []

for col in remarks_columns:
    condition = gas[col].astype(str).str.contains(
        r'elim|descar',
        case=False,
        na=False,
        regex=True
    )
    conditions.append(condition)

# Combine the remarks conditions using OR logic (any() along axis=1)
# Create a boolean series where True means 'elim' or 'descar' is found in at least one remark column
has_keywords = pd.concat(conditions, axis=1).any(axis=1)

# New condition: check if 'methane_intensity' is NaN
is_methane_intensity_null = gas['methane_intensity'].isna()

# New condition: check if 'undigested_dm_g' is NaN
is_undigested_dm_g_null = gas['undigested_dm_g'].isna()

# Combine all conditions for deletion using OR logic
final_delete_condition = has_keywords | is_methane_intensity_null | is_undigested_dm_g_null
#

# Create the 'delete' column, assigning 'yes' or 'no' based on the final condition
gas['delete'] = np.where(final_delete_condition, 'yes', 'no')

In [83]:
gas_clean = gas[gas['delete'] == 'no']

In [84]:
gas.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_complete_subsets_1234_2026_06_09.csv', index=None)

In [85]:
gas_clean.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_clean_subsets_1234_2026_06_09.csv', index=None)

In [86]:
nutrition.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/nutrition_complete_1234_2026_06_09.csv', index=None)

# 5.0 Gas

## 5.1 Mean

In [87]:
gas_clean.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete'],
      dtype='object')

In [88]:
#Columns processing
category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete']
numeric_columns = [ 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm']

for df in [gas_clean]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

/tmp/ipykernel_26244/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype('category')
/tmp/ipykernel_26244/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype('category')
/tmp/ipykernel_26244/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stab

In [89]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_gas'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [90]:
gas_clean_av = mean_with_replicates(gas_clean, category_columns, numeric_columns)

/tmp/ipykernel_26244/1161671181.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_26244/1161671181.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_26244/1161671181.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [91]:
gas_clean_av2 = gas_clean_av[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'net_gas_8h_ml','net_gas_24h_ml','gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas']]

In [92]:
gas_clean_av2

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h,n_replicates_gas
0,Dieta-1_Exp1,Dieta-1-Exp-1,3,839,LMF-invivo,Dieta 1_Exp1,NaN,30.67,67.46,149.92,34.90,2.31,14.42,14.69,21.82,71.84,321.88,6
1,Dieta-1_Exp2,Dieta-1-Exp-2,3,1019,LMF-invivo,Dieta 1_Exp2,NaN,28.26,57.51,129.22,33.33,2.41,13.66,14.43,18.49,70.34,313.12,8
2,Dieta-2_Exp1,Dieta-2-Exp-1,3,738,LMF-invivo,Dieta 2_Exp1,NaN,29.60,62.15,139.52,30.16,2.16,13.87,14.36,20.02,71.36,317.83,6
3,Dieta-2_Exp2,Dieta-2-Exp-2,3,1022,LMF-invivo,Dieta 2_Exp2,NaN,23.63,47.33,106.30,20.63,1.92,13.89,14.47,15.36,81.90,364.69,7
4,F24-3416,CIAT-705,1,1,Genetic_bank,Indigofera suffruticosa,Herbaceous_legumes,29.35,61.94,134.59,74.34,5.52,14.26,17.35,23.36,31.84,149.32,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
689,F26-0037,CIAT-18947,4,1048,Genetic_bank,Sesbania sesban,Shrub_Trees,35.23,73.90,154.75,51.99,3.36,13.67,13.76,21.30,41.93,-0.47,7
690,F26-0038,CIAT-19165,4,1051,Genetic_bank,Sesbania keniensis,Shrub_Trees,36.44,76.11,157.76,44.03,2.79,14.66,15.21,24.00,56.24,-8.23,7
691,F26-0039,CIAT-21899,4,1054,Genetic_bank,Sesbania sesban,Shrub_Trees,36.09,68.90,143.71,36.07,2.51,14.06,14.81,21.29,61.07,0.43,7
692,F26-0040,CIAT-23414,4,1057,Genetic_bank,Codariocalyx motorius,Shrub_Trees,29.68,59.26,122.99,26.23,2.13,14.60,15.26,18.76,73.03,-0.58,5


In [93]:
gas_clean_av2.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_clean_average_subsets_1234_2026_06_09.csv', index=None)

## 5.2 Standard deviation

In [94]:
# Group by subset and id_lab, keeping categorical columns
def std_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the standard deviation of the numeric columns
    std_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .std()
          .round(2)
    )

    # Count the number of replicates in each group
    std_df['n_replicates_gas'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, stds, and replicate counts
    result = (
        cat_df
        .join(std_df)
        .reset_index()
    )

    return result

In [95]:
gas_clean_std = std_with_replicates(gas_clean, category_columns, numeric_columns)

/tmp/ipykernel_26244/2013314210.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_26244/2013314210.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_26244/2013314210.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [96]:
gas_clean_std2 = gas_clean_std[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'net_gas_8h_ml','net_gas_24h_ml','gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas']]

In [97]:
gas_clean_std2.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_clean_std_subsets_1234_2026_06_09.csv', index=None)

#6.0 Nutrition

## 6.1 Mean

In [98]:
nutrition.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm'],
      dtype='object')

In [99]:
#Columns processing

category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat']
numeric_columns = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']

for df in [nutrition]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [100]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_nutrition'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [101]:
nutrition_av = mean_with_replicates(nutrition, category_columns, numeric_columns)

/tmp/ipykernel_26244/1194620962.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_26244/1194620962.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_26244/1194620962.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [102]:
nutrition_av.columns

Index(['id_lab', 'subset', 'no', 'requisitioner', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm', 'n_replicates_nutrition'],
      dtype='object')

In [103]:
nutrition_av2 = nutrition_av[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm', 'n_replicates_nutrition']]

In [104]:
nutrition_av2.head()

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_nutrition
0,Dieta-1_Exp1,Dieta-1-Exp-1,3,193.0,LMF-invivo,Dieta 1_Exp1,NaN,89.00,9.50,89.53,17.68,35.48,71.52,2
1,Dieta-1_Exp2,Dieta-1-Exp-2,3,235.0,LMF-invivo,Dieta 1_Exp2,NaN,89.00,9.50,89.53,0.00,0.00,0.00,2
2,Dieta-2_Exp1,Dieta-2-Exp-1,3,195.0,LMF-invivo,Dieta 2_Exp1,NaN,89.00,9.10,89.58,13.79,33.91,67.86,2
3,Dieta-2_Exp2,Dieta-2-Exp-2,3,237.0,LMF-invivo,Dieta 2_Exp2,NaN,89.00,9.10,89.58,0.00,0.00,0.00,2
4,F24-3416,CIAT-705,1,1.0,Genetic_bank,Indigofera suffruticosa,Herbaceous_legumes,91.97,12.68,87.32,33.01,28.05,45.22,2


In [105]:
nutrition_av2.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/nutrition_average_1234_2026_06_09.csv', index=None)

## 6.2 Standard deviation

In [106]:
# Group by subset and id_lab, keeping categorical columns
def std_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the standard deviation of the numeric columns
    std_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .std()
          .round(2)
    )

    # Count the number of replicates in each group
    std_df['n_replicates_nutrition'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, stds, and replicate counts
    result = (
        cat_df
        .join(std_df)
        .reset_index()
    )

    return result

In [107]:
nutrition_std = std_with_replicates(nutrition, category_columns, numeric_columns)

/tmp/ipykernel_26244/1995383719.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_26244/1995383719.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_26244/1995383719.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [108]:
nutrition_std2 = nutrition_std[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm', 'n_replicates_nutrition']]

In [109]:
nutrition_std2

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_nutrition
0,Dieta-1_Exp1,Dieta-1-Exp-1,3,193.0,LMF-invivo,Dieta 1_Exp1,NaN,0.00,0.00,0.00,0.00,0.00,0.00,2
1,Dieta-1_Exp2,Dieta-1-Exp-2,3,235.0,LMF-invivo,Dieta 1_Exp2,NaN,0.00,0.00,0.00,0.00,0.00,0.00,2
2,Dieta-2_Exp1,Dieta-2-Exp-1,3,195.0,LMF-invivo,Dieta 2_Exp1,NaN,0.00,0.00,0.00,0.00,0.00,0.00,2
3,Dieta-2_Exp2,Dieta-2-Exp-2,3,237.0,LMF-invivo,Dieta 2_Exp2,NaN,0.00,0.00,0.00,0.00,0.00,0.00,2
4,F24-3416,CIAT-705,1,1.0,Genetic_bank,Indigofera suffruticosa,Herbaceous_legumes,0.06,0.07,0.07,0.00,0.62,0.02,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
663,F26-0038,CIAT-19165,4,NaN,Genetic_bank,Sesbania keniensis,Shrub_Trees,0.06,2.68,2.68,0.00,0.38,0.07,2
664,F26-0039,CIAT-21899,4,NaN,Genetic_bank,Sesbania sesban,Shrub_Trees,0.06,0.04,0.04,0.00,0.55,0.20,2
665,F26-0040,CIAT-23414,4,NaN,Genetic_bank,Codariocalyx motorius,Shrub_Trees,0.23,0.14,0.14,0.46,1.55,2.07,2
666,F26-0041,CIAT-23767,4,NaN,Genetic_bank,Desmodium nicaraguense,Shrub_Trees,0.06,0.07,0.07,0.00,0.90,1.68,2


In [110]:
nutrition_std2.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/nutrition_std_1234_2026_06_09.csv', index=None)

# 7.0 Quality Check

## 7.1 Cleaning functions

Number of id_labs by subset in gas dataframe

In [111]:
# Function to clean and standardize lab/sample IDs to FXX-XXXX format
def clean_lab_ids(series):
    return (
        series
        .astype(str)
        .str.replace(r'\s+', '-', regex=True)                 # Replace spaces with -
        .str.replace(r'^(F\d{2})(\d+)', r'\1-\2', regex=True) # Format FXX-XXXX
    )


# Usage:
# subset_1_information_samples['10_ciat_lab_id'] = clean_lab_ids(subset_1_information_samples['10_ciat_lab_id'])

## 7.2 Import data
(Isabel averaged samples)

In [112]:
isa_subset_1 = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/isa_counts/isa_subset_1.csv')
isa_subset_2 = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/isa_counts/isa_subset_2.csv')
isa_subset_3 = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/isa_counts/isa_subset_3.csv')
isa_subset_4 = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/isa_counts/isa_subset_4.csv')

## 7.3 Formating

In [113]:
isa_subset_1['id_lab'] = clean_lab_ids(isa_subset_1['id_lab'])
isa_subset_2['id_lab'] = clean_lab_ids(isa_subset_2['id_lab'])
isa_subset_3['id_lab'] = clean_lab_ids(isa_subset_3['id_lab'])
isa_subset_4['id_lab'] = clean_lab_ids(isa_subset_4['id_lab'])

## 7.4 Database sample counts by subset

In [114]:
subset_1 = gas[gas['subset'] == 1]
subset_1_clean = gas_clean[gas_clean['subset'] == 1]

In [115]:
subset_2 = gas[gas['subset'] == 2]
subset_2_clean = gas_clean[gas_clean['subset'] == 2]

In [116]:
subset_3 = gas[gas['subset'] == 3]
subset_3_clean = gas_clean[gas_clean['subset'] == 3]

In [117]:
subset_4 = gas[gas['subset'] == 4]
subset_4_clean = gas_clean[gas_clean['subset'] == 4]

## 7.5 Comparison between database sample counts and isabel average counts

In [118]:
print('-'*30)
print('     COUNTS BEFORE FILTERING')
print('          Database    Isabel ')
print('Subset 1:', '  ', subset_1.id_lab.nunique(), '     ', isa_subset_1.id_lab.nunique())
print('Subset 2:', '  ', subset_2.id_lab.nunique(), '     ', isa_subset_2.id_lab.nunique() )
print('Subset 3:', '   ', subset_3.id_lab.nunique(), '     ', isa_subset_3.id_lab.nunique() )
print('Subset 4:', '  ', subset_4.id_lab.nunique(), '     ', isa_subset_4.id_lab.nunique() )
print('-'*30)
print('     COUNTS AFTER FILTERING')
print('          Database    Isabel ')
print('Subset 1:', '  ', subset_1_clean.id_lab.nunique(), '     ', isa_subset_1.id_lab.nunique())
print('Subset 2:', '  ', subset_2_clean.id_lab.nunique(), '     ', isa_subset_2.id_lab.nunique() )
print('Subset 3:', '   ', subset_3_clean.id_lab.nunique(), '     ', isa_subset_3.id_lab.nunique() )
print('Subset 4:', '  ', subset_4_clean.id_lab.nunique(), '     ', isa_subset_4.id_lab.nunique() )
print('-'*30)

------------------------------
     COUNTS BEFORE FILTERING
          Database    Isabel 
Subset 1:    209       232
Subset 2:    245       233
Subset 3:     128       128
Subset 4:    144       144
------------------------------
     COUNTS AFTER FILTERING
          Database    Isabel 
Subset 1:    209       232
Subset 2:    224       233
Subset 3:     127       128
Subset 4:    139       144
------------------------------


##7.6 Duplicates at Isabel average counts

In [119]:
repeated_isa_subset_1 = (isa_subset_1["id_lab"].value_counts().loc[lambda x: x > 1].to_frame("n_replicates").reset_index(names="id_lab"))
repeated_isa_subset_2 = (isa_subset_2["id_lab"].value_counts().loc[lambda x: x > 1].to_frame("n_replicates").reset_index(names="id_lab"))
repeated_isa_subset_3 = (isa_subset_3["id_lab"].value_counts().loc[lambda x: x > 1].to_frame("n_replicates").reset_index(names="id_lab"))
repeated_isa_subset_4 = (isa_subset_4["id_lab"].value_counts().loc[lambda x: x > 1].to_frame("n_replicates").reset_index(names="id_lab"))

In [120]:
print('Repeated samples in Isabel database')
print('             Samples    Repetitions')
print('Subset 1:      ', repeated_isa_subset_1.shape[0], '        ',  repeated_isa_subset_1["n_replicates"].sum())
print('Subset 2:       ', repeated_isa_subset_2.shape[0], '        ',  repeated_isa_subset_2["n_replicates"].sum())
print('Subset 3:       ', repeated_isa_subset_3.shape[0], '        ',  repeated_isa_subset_3["n_replicates"].sum())
print('Subset 4:       ', repeated_isa_subset_4.shape[0], '         ',  repeated_isa_subset_4["n_replicates"].sum())

Repeated samples in Isabel database
             Samples    Repetitions
Subset 1:       36          86
Subset 2:        3          24
Subset 3:        3          11
Subset 4:        1           5


In [121]:
list_repeated_isa_subset_1 = repeated_isa_subset_1.id_lab.to_list()
print('Repeated samples:',len(list_repeated_isa_subset_1))
print(list_repeated_isa_subset_1)

Repeated samples: 36
['F25-0017', 'F25-0018', 'F25-0019', 'F24-3427', 'F24-3431', 'F24-3428', 'F24-3429', 'F24-3430', 'F24-3443', 'F24-3444', 'F24-3445', 'F24-3446', 'F24-3447', 'F24-3448', 'F24-3449', 'F24-3442', 'F24-3434', 'F24-3441', 'F24-3440', 'F24-3439', 'F24-3438', 'F24-3437', 'F24-3436', 'F24-3435', 'F24-3450', 'F24-3453', 'F24-3452', 'F24-3455', 'F24-3451', 'F24-3432', 'F24-3433', 'F24-3426', 'F24-3454', 'F25-0848', 'F25-0845', 'F25-0873']


In [122]:
list_repeated_isa_subset_2 = repeated_isa_subset_2.id_lab.to_list()
print('Repeated samples:',len(list_repeated_isa_subset_2))
print(list_repeated_isa_subset_2)

Repeated samples: 3
['F25-0008', 'F25-0017', 'F25-0018']


In [123]:
list_repeated_isa_subset_3 = repeated_isa_subset_3.id_lab.to_list()
print('Repeated samples:',len(list_repeated_isa_subset_3))
print(list_repeated_isa_subset_3)

Repeated samples: 3
['F25-0008', 'F25-0017', 'F25-1667']


In [124]:
list_repeated_isa_subset_4 = repeated_isa_subset_4.id_lab.to_list()
print('Repeated samples:',len(list_repeated_isa_subset_4))
print(list_repeated_isa_subset_4)

Repeated samples: 1
['F25-0008']


# 7.7 Absent samples

In [125]:
list_subset_1 = subset_1["id_lab"].unique().tolist()
list_subset_2 = subset_2["id_lab"].unique().tolist()
list_subset_3 = subset_3["id_lab"].unique().tolist()
list_subset_4 = subset_4["id_lab"].unique().tolist()

In [126]:
list_isa_subset_1 = isa_subset_1["id_lab"].unique().tolist()
list_isa_subset_2 = isa_subset_2["id_lab"].unique().tolist()
list_isa_subset_3 = isa_subset_3["id_lab"].unique().tolist()
list_isa_subset_4 = isa_subset_4["id_lab"].unique().tolist()

In [127]:
absent_in_subset_1 = list(set(list_isa_subset_1) - set(list_subset_1))
print(len(absent_in_subset_1))
print(absent_in_subset_1)

23
['F25-0783', 'F25-0861', 'F24-1075', 'F25-0772', 'F25-0914', 'F25-0904', 'F24-1070', 'F25-0825', 'F25-0913', 'F25-0788', 'F25-0824', 'F25-0839', 'F24-1095', 'F25-0837', 'F25-0926', 'F24-1084', 'F25-0820', 'F25-0868', 'F25-0813', 'F24-1047', 'F25-0823', 'F25-0784', 'F25-0908']


In [128]:
absent_in_subset_2 = list(set(list_isa_subset_2) - set(list_subset_2))
print(len(absent_in_subset_2))
print(absent_in_subset_2)
#Estas son las que se eliminan porque estan duplicadas en el subset 2 y 3

9
['F25-1654', 'F25-1656', 'F25-1665', 'F25-1664', 'F25-1653', 'F25-1658', 'F25-1655', 'F25-1652', 'F25-1651']


In [129]:
absent_in_subset_3 = list(set(list_isa_subset_3) - set(list_subset_3))
print(len(absent_in_subset_3))
print(absent_in_subset_3)

0
[]


In [130]:
absent_in_subset_4 = list(set(list_isa_subset_4) - set(list_subset_4))
print(len(absent_in_subset_4))
print(absent_in_subset_4)

0
[]


## 7.8 MANUAL CHECK ON ISABEL FILES:

### Subset 1


F25-0772: Tab 3.1 present (Only Run 1) - Tab 4.0  absent - Tab 4.1av present - id: ICARDA-165563 - tax_name: Trifolium pratense

F25-0825: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-6984-4-119709 - tax_name: Medicago sativa

F25-0913: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: **CIAT-1853** - tax_name: Stylosanthes guianensis **Absent in other subsets**

F24-1047: Tab 3.1 present (**Extrange tddm**) - Tab 4.0 absent - Tab 4.1av present - id: LMF - tax_name: Albizzia saman

F25-0820: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ICARDA-T216010

F24-1084: Tab 3.1 present (**Extrange tddm**) - Tab 4.0 absent - Tab 4.1av present - id: LMF - tax_name: Brachiaria decumbens

F25-0839: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-7049-18-119723 - tax_name: Trifolium repens

F25-0837: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id:ILRI-10920-16-119721 - tax_name: Arachis pintoi

F25-0861: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-14539-40-119745 - tax_name: Centrosema virginianum

F25-0823: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-6529-2-119707 - tax_name: Labalab purpureues

F24-1075: Tab 3.1 present (**Extrange tddm**) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-6529-2-119707 - tax_name: Labalab purpureues

F25-0926: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: **CIAT-6818** - tax_name: Panicum maximum **Absent in other subsets**

F25-0824: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-10925-3-119708 - tax_name: Arachis glabrata


F25-0904: Tab 3.1 present (Only Run 1) - Tab 4.0 **present** - Tab 4.1av present - id: **CIAT-22752** - tax_name: Labalab purpureues **Absent in other subsets**

F25-0868: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ILRI-70-47-120164 - tax_name: Leucaena leucocephala


F25-0813: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ICARDA-T216003 - tax_name: Rhus tripartita (Ucria) Grande

F24-1095: Tab 3.1 present (**Extrange tddm**) - Tab 4.0 absent - Tab 4.1av present - id: LMF - tax_name: Setaria sphacelata

F25-0908: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: **CIAT-22192** - tax_name: Leucaena diversifolia **Absent in other subsets**

F24-1070: Tab 3.1 present (**Extrange tddm**) - Tab 4.0 absent - Tab 4.1av present - id: LMF - tax_name: Gliricidia sepium

F25-0914: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: **CIAT-10625** - tax_name: Stylosanthes scabra **Absent in other subsets**


F25-0788: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ICARDA-J13 - tax_name: Cichorium pumilum


F25-0783: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ICARDA-J8 - tax_name: Daucus carota

F25-0784: Tab 3.1 present (Only Run 1) - Tab 4.0 absent - Tab 4.1av present - id: ICARDA-J8 - tax_name: Daucus carota



### Subset 2

['F25-1652', 'F25-1658', 'F25-1665', 'F25-1655', 'F25-1651', 'F25-1654', 'F25-1656', 'F25-1653', 'F25-1664'] Deleated because were reprocessed on subset 3



